# 07 — CP1 B2: protocol và generator tổng hợp

Notebook độc lập; chạy từ cell đầu, không cần thực thi lại NB06.

## 1. Thiết lập B2 (b2-synthetic-v1)

Đọc `reports/cp1/benchmark_protocol.md` trước lượt đầu. Hai job: preparation đọc đủ cleaned dataset
một lần, kiểm mọi nguồn B1, cache 5 nguồn/user và khóa phép chia/cohort; generation chỉ sinh phần
selection. Cả hai mặc định tắt. Không chạy detector ở B2.

Matrix có 36 ô chính + 35 ô stress/bổ sung = 71 ô, 5 seeds/user/ô. Các user có bản sao nguồn y hệt
cùng một nửa; source bất thường chỉ bị cách ly khỏi generator, không xóa GPS gốc. Từ cohort mới biết
chế độ B4; nếu thiếu cohort thì giữ baseline và không có lượt final.


In [ ]:
from pathlib import Path
import sys, json, importlib
import pandas as pd
from IPython.display import display
ROOT_B2 = Path.cwd().resolve()
if not (ROOT_B2 / 'pyproject.toml').exists():
    ROOT_B2 = ROOT_B2.parent
assert (ROOT_B2 / 'reports/cp1/benchmark_protocol.md').exists()
if str(ROOT_B2 / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT_B2 / 'src'))
import geolife_inference.benchmark_synthetic as synthetic
synthetic = importlib.reload(synthetic)
PREP_B2 = ROOT_B2 / 'artifacts/benchmark/b2/prepared_v1'
CORPUS_B2 = ROOT_B2 / 'artifacts/benchmark/b2/selection_v1'
RUN_B2_PREPARE = False
RUN_B2_GENERATE = True
COMPUTE_USED_B2_SECONDS = 3600.0  # Tam ghi no 1 gio ngan sach cho cac luot truoc; khong phai runtime da do.
print('B2 module:', synthetic.__file__)
print('Preparation:', PREP_B2)
print('Selection corpus:', CORPUS_B2)


B2 module: E:\QUY\Geolife-Trajectories\src\geolife_inference\benchmark_synthetic.py
Preparation: E:\QUY\Geolife-Trajectories\artifacts\benchmark\b2\prepared_v1
Selection corpus: E:\QUY\Geolife-Trajectories\artifacts\benchmark\b2\selection_v1


## 2. Chuẩn bị nguồn và khóa protocol

Điền COMPUTE_USED_B2_SECONDS bằng số đã ghi nhận hoặc tổng bảo thủ nếu lượt cũ thiếu timing;
137,18 giây profile và 864,70 giây CLI đo riêng chỉ là hai số đã biết, không phải toàn bộ ledger.
Bật RUN_B2_PREPARE cho đúng một lượt. Namespace đã có sẽ bị từ chối, kể cả failed; đọc audit trước
khi tạo version mới. Giữ trần 24 giờ tích lũy, 12 giờ/job và RAM ≤ min(8 GiB, 50% khả dụng đầu job).


In [2]:
if RUN_B2_PREPARE:
    if COMPUTE_USED_B2_SECONDS is None:
        raise ValueError('Điền tổng compute đã dùng trước khi khởi job B2')
    budget_b2 = synthetic.Budget(float(COMPUTE_USED_B2_SECONDS), synthetic.memory_usage, synthetic.available_memory)
    protocol_b2 = synthetic.prepare(ROOT_B2, PREP_B2, budget_b2)
    print('Preparation completed:', protocol_b2['mode'])
else:
    print('Preparation tắt; chưa quét cleaned dataset.')


E:\QUY\Geolife-Trajectories\src\geolife_inference\benchmark_synthetic.py:188: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(part.timestamp, utc=True).astype("int64").to_numpy() / 1e9
E:\QUY\Geolife-Trajectories\src\geolife_inference\benchmark_synthetic.py:188: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(part.timestamp, utc=True).astype("int64").to_numpy() / 1e9
E:\QUY\Geolife-Trajectories\src\geolife_inference\benchmark_synthetic.py:188: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(part.timestamp, utc=True).astype(

Preparation completed: selection_confirmation_cohort_pending


In [3]:
protocol_path_b2 = PREP_B2 / 'protocol.json'
if protocol_path_b2.exists():
    prep_audit_b2 = json.loads((PREP_B2 / 'preparation_audit.json').read_text(encoding='utf-8'))
    if prep_audit_b2['status'] != 'completed':
        raise ValueError('Preparation chưa completed; không dùng partial corpus')
    protocol_b2 = json.loads(protocol_path_b2.read_text(encoding='utf-8'))
    cohort_b2 = pd.read_csv(PREP_B2 / 'cohort.csv', dtype={'user_id':str})
    display(pd.DataFrame(protocol_b2['feasibility']).T)
    display(cohort_b2)
    matrix_b2 = pd.DataFrame(synthetic.cells(protocol_b2['stress_rms_m']))
    display(matrix_b2.groupby('group').size().rename('cells').to_frame())
    print('Mode:', protocol_b2['mode'], '| Stress RMS:', protocol_b2['stress_rms_m'], 'm')
    display(pd.DataFrame(protocol_b2['volume_plan']).T)
    print('Case quota per split:', {s:sum(v==s for v in protocol_b2['split'].values())*len(matrix_b2)*5 for s in ['selection','final']})
    print('Compute sau preparation:', prep_audit_b2['compute_used_after_seconds'], 'giây; cộng mọi job mới khác trước generation.')
    print('Radius cohort còn unknown: không suy đủ baseline là (c) đã đạt.')
else:
    print('Chưa có protocol/source manifest đã khóa. Chạy mục 2 trước.')


,source_users,baseline_at_least_5,independent_source_groups,negative_denominator_users,common,radius_edge_cohort
selection,81,57,66,18,16,unknown: per-user radius outputs not saved
final,81,54,66,25,17,unknown: per-user radius outputs not saved


,user_id,split,independence_group,cached_sources,baseline_stays,baseline_at_least_5,negative_denominator_positive,valid_label_seconds,baseline_labeled_stay_seconds,baseline_motor_overlap_seconds
0,000,selection,4e517985fe7c4ab3,5,120,True,False,NaN,NaN,NaN
1,001,selection,d5dc721078d2251a,5,44,True,False,NaN,NaN,NaN
2,002,final,22e70249669b6e2f,5,188,True,False,NaN,NaN,NaN
3,003,selection,4e517985fe7c4ab3,5,436,True,False,NaN,NaN,NaN
4,004,final,f3b82e5195e28dc8,5,224,True,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
177,177,excluded_no_eligible_source,NaN,0,0,False,False,NaN,NaN,NaN
178,178,excluded_no_eligible_source,NaN,0,0,False,False,NaN,NaN,NaN
179,179,selection,ab1644a0ec3f1024,5,56,True,True,315630.0,17881.0,1008.0
180,180,final,be22b1076648e6a4,5,9,True,False,NaN,NaN,NaN


,cells
group,
dropout,12
main,36
negative,4
stress,12
two_stops,6
unobservable,1


Mode: selection_confirmation_cohort_pending | Stress RMS: 100 m


,cases,sampled_fixes_upper_bound,runtime
selection,28755,53514998,unmeasured
final,28755,43478839,unmeasured


Case quota per split: {'selection': 28755, 'final': 28755}
Compute sau preparation: 3690.758348100004 giây; cộng mọi job mới khác trước generation.
Radius cohort còn unknown: không suy đủ baseline là (c) đã đạt.


## 3. Sinh phần selection — chưa chấm detector

Sau khi đọc cohort/matrix/mode, bật RUN_B2_GENERATE. Dữ liệu final không được sinh trong notebook
ở giai đoạn này. Generator dùng các point nguồn làm polyline latent, chèn stop và tạo clock/noise/
dropout độc lập detector. File observations không chứa nhãn stop; truth có latent path, source IDs,
event start/end/center và counts kể cả case mất hết observations.

Nếu kernel đã chạy thêm job từ sau preparation, cập nhật COMPUTE_USED_B2_SECONDS lên tổng mới.
Không giảm số user/cell/seed để làm audit completed khi gặp giới hạn.


In [4]:
if RUN_B2_GENERATE:
    prep_audit_b2 = json.loads((PREP_B2 / 'preparation_audit.json').read_text(encoding='utf-8'))
    if prep_audit_b2['status'] != 'completed':
        raise ValueError('Preparation chưa completed')
    used_b2 = max(float(COMPUTE_USED_B2_SECONDS or 0), prep_audit_b2['compute_used_after_seconds'])
    generation_budget_b2 = synthetic.Budget(used_b2, synthetic.memory_usage, synthetic.available_memory)
    result_b2 = synthetic.generate(PREP_B2, CORPUS_B2, 'selection', generation_budget_b2)
    print(result_b2)
else:
    print('Generation tắt; chưa sinh corpus.')


Generation tắt; chưa sinh corpus.


## 4. Đọc audit B2 và điều kiện chuyển bước

Chỉ completed khi đủ mọi user trong split × 71 ô × 5 seed, input/cache/hash đúng và không vượt
budget. Không tick B2 chỉ vì notebook chạy import. Các lỗi dữ liệu/thiết kế phải sửa ở version mới
trước tuning. Không xem final để sửa code. B3 audit cleaning theo PLAN; B4 mới chạy fixed-anchor,
median-updated và ST-DBSCAN. Detectors chạy từng case độc lập, macro theo source user; không loại
stays chồng thời gian giữa các case tổng hợp dùng cùng ngày gốc.


In [5]:
generation_audit_b2 = CORPUS_B2 / 'generation_audit.json'
if generation_audit_b2.exists():
    display(pd.Series(json.loads(generation_audit_b2.read_text(encoding='utf-8'))))
else:
    print('Chưa có generation audit. B2 vẫn pending run.')


Chưa có generation audit. B2 vẫn pending run.
